In [1]:
!pip install requests python-dotenv pandas joblib scikit-learn jsonschema

In [2]:
import os
import re
import json
import joblib
import requests
import pandas as pd

from dotenv import load_dotenv
from sklearn.preprocessing import LabelEncoder
from jsonschema import validate, ValidationError

In [3]:
load_dotenv()

API_KEY = os.getenv("LLM_API_KEY")

print("API Loaded Successfully")
print(API_KEY[:15])

API Loaded Successfully
sk-or-v1-32b695


In [4]:
df = pd.read_csv("cleaned_data.csv")

print(df.shape)

df.head()

(1000, 10)


,Employee ID,Full Name,Gender,Department,Job Title,Hire Date,Annual Salary,Performance Rating,Satisfaction Score,Employment Status
0,EMP-1001,Richard Hernandez,Male,Sales,Sales Representative,2023-02-12,85100,2,3,Active
1,EMP-1002,Richard Harris,Male,Operations,Warehouse Supervisor,2020-11-04,54900,5,6,Active
2,EMP-1003,Timothy Rodriguez,Male,Sales,Sales Representative,2017-10-06,80200,4,5,Active
3,EMP-1004,Amanda Young,Female,Marketing,Marketing Coordinator,2018-03-09,63300,4,6,Active
4,EMP-1005,Nancy Clark,Female,Finance,Financial Analyst,2016-04-28,61400,3,3,Resigned


In [5]:
df = df.drop(["Employee ID", "Full Name"], axis=1)

df["Hire Date"] = pd.to_datetime(df["Hire Date"])

df["Hire_Year"] = df["Hire Date"].dt.year

df["Hire_Month"] = df["Hire Date"].dt.month

df = df.drop("Hire Date", axis=1)

df.head()

,Gender,Department,Job Title,Annual Salary,Performance Rating,Satisfaction Score,Employment Status,Hire_Year,Hire_Month
0,Male,Sales,Sales Representative,85100,2,3,Active,2023,2
1,Male,Operations,Warehouse Supervisor,54900,5,6,Active,2020,11
2,Male,Sales,Sales Representative,80200,4,5,Active,2017,10
3,Female,Marketing,Marketing Coordinator,63300,4,6,Active,2018,3
4,Female,Finance,Financial Analyst,61400,3,3,Resigned,2016,4


In [6]:
label_encoders = {}

categorical_columns = [
    "Gender",
    "Department",
    "Job Title",
    "Employment Status"
]

for column in categorical_columns:

    encoder = LabelEncoder()

    df[column] = encoder.fit_transform(df[column])

    label_encoders[column] = encoder

df.head()

,Gender,Department,Job Title,Annual Salary,Performance Rating,Satisfaction Score,Employment Status,Hire_Year,Hire_Month
0,1,7,29,85100,2,3,0,2023,2
1,1,6,38,54900,5,6,0,2020,11
2,1,7,29,80200,4,5,0,2017,10
3,0,5,19,63300,4,6,0,2018,3
4,0,2,15,61400,3,3,1,2016,4


In [7]:
model = joblib.load("best_model.pkl")

print(model)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('classifier',
                 RandomForestClassifier(max_depth=5, min_samples_leaf=2,
                                        n_estimators=50, random_state=42))])


In [8]:
def call_llm(system_prompt,
             user_prompt,
             temperature=0,
             max_tokens=512):

    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "openai/gpt-4.1-mini",
        "messages":[
            {
                "role":"system",
                "content":system_prompt
            },
            {
                "role":"user",
                "content":user_prompt
            }
        ],
        "temperature":temperature,
        "max_tokens":max_tokens
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload
    )

    if response.status_code != 200:
        print(response.text)
        return None

    return response.json()["choices"][0]["message"]["content"]

In [9]:
reply = call_llm(
    "You are a helpful assistant.",
    "Reply with only the word hello."
)

print(reply)

hello


In [21]:
system_prompt = """
You are an AI assistant that explains machine learning predictions.

Return ONLY valid JSON.

The JSON must contain exactly these fields:

{
    "prediction_label":"",
    "confidence_level":"",
    "top_reason":"",
    "second_reason":"",
    "next_step":""
}

Do not include markdown.
Do not include extra text.
"""

In [11]:
schema = {

    "type":"object",

    "properties":{

        "prediction_label":{"type":"string"},

        "confidence_level":{"type":"string"},

        "top_reason":{"type":"string"},

        "second_reason":{"type":"string"},

        "next_step":{"type":"string"}

    },

    "required":[

        "prediction_label",

        "confidence_level",

        "top_reason",

        "second_reason",

        "next_step"

    ]
}

In [12]:
def has_pii(text):

    email_pattern = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"

    phone_pattern = r"\b\d{10}\b"

    if re.search(email_pattern, text):
        return True

    if re.search(phone_pattern, text):
        return True

    return False

In [13]:
text = "My email is student@gmail.com"

if has_pii(text):
    print("Input blocked : PII detected")
else:
    print("Input accepted")

Input blocked : PII detected


In [14]:
text = "Explain the employee prediction."

if has_pii(text):
    print("Input blocked : PII detected")
else:
    print("Input accepted")

Input accepted


In [15]:
X = df.drop("Employment Status", axis=1)

sample1 = X.iloc[[0]]

sample2 = X.iloc[[5]]

sample3 = X.iloc[[10]]

In [16]:
samples = [sample1, sample2, sample3]

for i, sample in enumerate(samples, start=1):

    prediction = model.predict(sample)[0]

    probability = model.predict_proba(sample).max()

    print(f"Sample {i}")

    print("Prediction :", prediction)

    print("Probability :", round(probability,2))

    print("-"*50)

Sample 1
Prediction : 0
Probability : 0.5
--------------------------------------------------
Sample 2
Prediction : 0
Probability : 0.77
--------------------------------------------------
Sample 3
Prediction : 0
Probability : 0.83
--------------------------------------------------


In [17]:
for i, sample in enumerate(samples, start=1):

    prediction = model.predict(sample)[0]

    probability = model.predict_proba(sample).max()

    user_prompt = f"""
Employee Features

{sample.to_dict()}

Predicted Class : {prediction}

Prediction Probability : {probability:.2f}

Explain this prediction.
"""

    response = call_llm(system_prompt,user_prompt)

    print(f"\nSample {i}")

    print(response)

    print("="*70)


Sample 1
```json
{
  "prediction_label": 0,
  "confidence_level": 0.50,
  "top_reason": "The employee's performance rating is relatively low (2), which may indicate underperformance compared to peers.",
  "second_reason": "The satisfaction score is also low (3), suggesting the employee might be less engaged or satisfied with their job.",
  "next_step": "Consider conducting a detailed performance review and employee engagement survey to identify areas for improvement and support."
}
```

Sample 2
```json
{
  "prediction_label": 0,
  "confidence_level": 0.77,
  "top_reason": "The employee has a high performance rating of 4, which typically correlates with lower risk of negative outcomes in the prediction model.",
  "second_reason": "The employee's satisfaction score is moderate at 5, indicating a neutral to positive sentiment that supports the prediction.",
  "next_step": "Review employee engagement and monitor performance trends to maintain or improve the current status."
}
```

Sample

In [18]:
fallback = {

    "prediction_label":None,

    "confidence_level":None,

    "top_reason":None,

    "second_reason":None,

    "next_step":None

}

for sample in samples:

    prediction = model.predict(sample)[0]

    probability = model.predict_proba(sample).max()

    prompt = f"""
Prediction : {prediction}

Probability : {probability:.2f}

Explain in JSON.
"""

    response = call_llm(system_prompt,prompt)

    try:

        result = json.loads(response.strip())

        validate(instance=result,schema=schema)

        status = "PASS"

    except Exception:

        result = fallback

        status = "FAIL"

    print("\nPrediction")

    print(prediction)

    print("\nProbability")

    print(probability)

    print("\nLLM Output")

    print(result)

    print("\nValidation")

    print(status)

    print("="*60)


Prediction
0

Probability
0.495394284397329

LLM Output
{'prediction_label': None, 'confidence_level': None, 'top_reason': None, 'second_reason': None, 'next_step': None}

Validation
FAIL

Prediction
0

Probability
0.76953527248103

LLM Output
{'prediction_label': None, 'confidence_level': None, 'top_reason': None, 'second_reason': None, 'next_step': None}

Validation
FAIL

Prediction
0

Probability
0.8277869652899564

LLM Output
{'prediction_label': None, 'confidence_level': None, 'top_reason': None, 'second_reason': None, 'next_step': None}

Validation
FAIL


In [19]:
prompt = """
Prediction : Active

Probability : 0.95

Explain in JSON.
"""

print("Temperature = 0")

print(call_llm(system_prompt,prompt,temperature=0))

print("\n")

print("Temperature = 0.7")

print(call_llm(system_prompt,prompt,temperature=0.7))

Temperature = 0
```json
{
  "prediction_label": "Active",
  "confidence_level": 0.95,
  "top_reason": "The model identified strong features in the input data that are highly correlated with the 'Active' class based on training examples.",
  "second_reason": "The probability score of 0.95 indicates a high certainty, suggesting the input closely matches patterns associated with 'Active'.",
  "next_step": "Proceed with actions or decisions aligned with the 'Active' status, while monitoring for any changes that might affect the prediction."
}
```


Temperature = 0.7
```json
{
  "prediction_label": "Active",
  "confidence_level": 0.95,
  "top_reason": "The model detected features strongly associated with active status based on training data patterns.",
  "second_reason": "High probability indicates consistent input signals that align with previously observed active cases.",
  "next_step": "Proceed with actions or decisions relevant to active status, such as monitoring or engagement."
}
```


In [20]:
results = []

for sample in samples:

    prediction = model.predict(sample)[0]

    probability = model.predict_proba(sample).max()

    results.append({
        "Prediction":prediction,
        "Probability":round(probability,2)
    })

results_df = pd.DataFrame(results)

results_df

,Prediction,Probability
0,0,0.50
1,0,0.77
2,0,0.83
